# AU-FISH V3 Extension: YOLOv10n Coral-Only Ablation

Notebook ini terpisah dari V3 utama dan dari extension YOLOv10n dua kelas.

**Sumber data, read-only**

```text
dataset_v2/experiments/multiseed_wbf_results_ready_v3_complete/
```

**Output baru**

```text
dataset_v2/experiments/
multiseed_wbf_results_ready_v3_extension_yolov10n_coral_only_v1/
```

Notebook hanya membuat JSON coral-only, label, checkpoint, prediction, dan report. Folder gambar tidak diduplikasi di Google Drive. Semua gambar tetap dipakai sehingga gambar tanpa anotasi coral menjadi negative images.

Scope:

- YOLOv10n
- satu kelas coral
- time-based dan source-based
- seed 42, 123, 2026
- 100 epoch, input 640, batch 16
- common COCOeval
- optional FPS
- optional perbandingan coral AP dengan extension YOLOv10n dua kelas

In [ ]:
#@title 1. Install
%pip -q install ultralytics==8.4.102 pycocotools pandas pyyaml pillow tqdm psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.8 MB/s eta 0:00:00


In [ ]:
#@title 2. Mount Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#@title 3. Configuration and paths
import zipfile

from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence
from collections import defaultdict
from contextlib import redirect_stdout
from io import StringIO
import os, re, gc, json, time, random, hashlib, shutil, subprocess, platform
import importlib.metadata as importlib_metadata
import numpy as np
import pandas as pd
import yaml

V3_TAG = "multiseed_wbf_results_ready_v3_complete"
TWO_CLASS_TAG = "multiseed_wbf_results_ready_v3_extension_yolov10n_v1"
EXTENSION_TAG = "multiseed_wbf_results_ready_v3_extension_yolov10n_coral_only_v1"

PROTOCOLS = ["time_based", "source_based"]
SPLITS = ["train", "val", "test"]
SEEDS = [42, 123, 2026]
MODEL_NAME = "yolov10n"
PRETRAINED_WEIGHTS = "/content/drive/MyDrive/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_yolov10n_v1/runs/time_based/yolov10n/seed_42/train/weights/best.pt"

TRAIN_CONFIG = {
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "workers": 0,
    "cache": False,
    "amp": True,
    "pretrained": True,
}

INFER_CONFIG = {
    "conf": 0.001,
    "iou": 0.70,
    "max_det": 300,
    "imgsz": 640,
    "batch": 8,
    "augment": False,
}

RUN_FILTER = True
RUN_TRAINING = True
RUN_EVALUATION = True
RUN_OPTIONAL_FPS = False
RUN_OPTIONAL_TWO_CLASS_COMPARISON = True

FPS_SEED = 42
FPS_IMAGES = 100
FPS_WARMUP = 10
FPS_REPEATS = 3

ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/au-fish"),
    Path("/content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish"),
]

def resolve_root():
    for candidate in ROOT_CANDIDATES:
        if (candidate / "dataset_v2").is_dir():
            return candidate.resolve()
    for search_root in [Path("/content/drive/MyDrive"), Path("/content/drive/.shortcut-targets-by-id")]:
        if not search_root.exists():
            continue
        for dataset_root in search_root.rglob("dataset_v2"):
            if dataset_root.is_dir() and all((dataset_root / p).is_dir() for p in PROTOCOLS):
                return dataset_root.parent.resolve()
    raise FileNotFoundError("Folder AU-FISH tidak ditemukan.")

PROJECT_ROOT = resolve_root()
DATASET_ROOT = PROJECT_ROOT / "dataset_v2"

V3_ROOT = DATASET_ROOT / "experiments" / V3_TAG
V3_CANONICAL = V3_ROOT / "canonical_data"
V3_MANIFESTS = V3_ROOT / "manifests"

TWO_CLASS_ROOT = DATASET_ROOT / "experiments" / TWO_CLASS_TAG

EXT_ROOT = DATASET_ROOT / "experiments" / EXTENSION_TAG
CANONICAL_ROOT = EXT_ROOT / "canonical_coral_only"
CONFIG_ROOT = EXT_ROOT / "config"
RUNS_ROOT = EXT_ROOT / "runs"
PRED_ROOT = EXT_ROOT / "predictions"
REPORTS_ROOT = EXT_ROOT / "reports"
MANIFESTS_ROOT = EXT_ROOT / "manifests"

for folder in [CANONICAL_ROOT, CONFIG_ROOT, RUNS_ROOT, PRED_ROOT, REPORTS_ROOT, MANIFESTS_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

LOCAL_IMAGES = Path("/content") / f"{EXTENSION_TAG}_images"
LOCAL_DATA = Path("/content") / f"{EXTENSION_TAG}_data"
LOCAL_RUNS = Path("/content") / f"{EXTENSION_TAG}_runs"
for folder in [LOCAL_IMAGES, LOCAL_DATA, LOCAL_RUNS]:
    folder.mkdir(parents=True, exist_ok=True)

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)

def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, ensure_ascii=False, default=str)
    tmp.replace(path)

def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def sample_sd(values):
    values = np.asarray(values, dtype=float)
    return float(np.std(values, ddof=1)) if len(values) > 1 else float("nan")

scope = {
    "extension_tag": EXTENSION_TAG,
    "parent_v3_tag": V3_TAG,
    "model": MODEL_NAME,
    "task": "coral_only",
    "protocols": PROTOCOLS,
    "seeds": SEEDS,
    "expected_training_runs": 6,
    "keeps_all_images": True,
    "removes_fish_annotations": True,
    "duplicates_images_in_drive": False,
    "writes_to_parent_v3": False,
}
atomic_json(scope, CONFIG_ROOT / "scope_contract.json")
print(json.dumps(scope, indent=2))

{
  "extension_tag": "multiseed_wbf_results_ready_v3_extension_yolov10n_coral_only_v1",
  "parent_v3_tag": "multiseed_wbf_results_ready_v3_complete",
  "model": "yolov10n",
  "task": "coral_only",
  "protocols": [
    "time_based",
    "source_based"
  ],
  "seeds": [
    42,
    123,
    2026
  ],
  "expected_training_runs": 6,
  "keeps_all_images": true,
  "removes_fish_annotations": true,
  "duplicates_images_in_drive": false,
  "writes_to_parent_v3": false
}


In [ ]:
#@title 4. Filter V3 canonical JSON to coral only

lock_path = V3_MANIFESTS / "canonical_dataset_lock.json"
if not lock_path.exists():
    raise FileNotFoundError(lock_path)

lock = load_json(lock_path)
if not lock.get("ready_for_training", False):
    raise RuntimeError("V3 canonical dataset belum terkunci.")

rows = []

for protocol in PROTOCOLS:
    for split in SPLITS:
        src = V3_CANONICAL / protocol / split / "annotations.json"
        source = load_json(src)

        category_map = {int(c["id"]): str(c["name"]) for c in source.get("categories", [])}
        if category_map != {1: "coral", 2: "fish"}:
            raise RuntimeError(f"Kategori V3 tidak sesuai: {src}")

        coral_annotations = [
            dict(ann) for ann in source.get("annotations", [])
            if int(ann["category_id"]) == 1
        ]

        for new_id, ann in enumerate(coral_annotations, start=1):
            ann["id"] = new_id

        output = {
            "info": {
                **source.get("info", {}),
                "extension_tag": EXTENSION_TAG,
                "task": "coral_only",
                "parent_v3_path": str(src),
                "parent_v3_sha256": sha256_file(src),
                "kept_all_images": True,
                "images_duplicated_in_drive": False,
            },
            "licenses": source.get("licenses", []),
            "images": source.get("images", []),
            "annotations": coral_annotations,
            "categories": [{"id": 1, "name": "coral"}],
        }

        dst = CANONICAL_ROOT / protocol / split / "annotations.json"
        if RUN_FILTER or not dst.exists():
            atomic_json(output, dst)

        annotated_ids = {int(a["image_id"]) for a in coral_annotations}

        rows.append({
            "protocol": protocol,
            "split": split,
            "source_path": str(src),
            "source_sha256": sha256_file(src),
            "output_path": str(dst),
            "output_sha256": sha256_file(dst),
            "images": len(output["images"]),
            "coral_annotations": len(coral_annotations),
            "annotated_images": len(annotated_ids),
            "negative_images": len(output["images"]) - len(annotated_ids),
            "fish_annotations": 0,
            "kept_all_images": True,
            "images_duplicated_in_drive": False,
        })

filter_manifest = pd.DataFrame(rows)
save_csv(filter_manifest, MANIFESTS_ROOT / "coral_only_filter_manifest.csv")
display(filter_manifest)

,protocol,split,source_path,source_sha256,output_path,output_sha256,images,coral_annotations,annotated_images,negative_images,fish_annotations,kept_all_images,images_duplicated_in_drive
0,time_based,train,/content/drive/.shortcut-targets-by-id/1kW1zea...,ce8d95f2f25ede1c5fbdb21bed94e8e8158440a5d878fb...,/content/drive/.shortcut-targets-by-id/1kW1zea...,9a69c8bef4df3eda69e73a174da95ee61df2a995f74e83...,1061,1788,686,375,0,True,False
1,time_based,val,/content/drive/.shortcut-targets-by-id/1kW1zea...,a361d742c1812bbb61ea7377e03ffb19e2db41020557f7...,/content/drive/.shortcut-targets-by-id/1kW1zea...,c8a49530caf42326a71325eb133dea5eb5f2e2614ab8b2...,203,280,126,77,0,True,False
2,time_based,test,/content/drive/.shortcut-targets-by-id/1kW1zea...,3a8bf5b2431f57615630b7228682428e4bb448d7bfb8f7...,/content/drive/.shortcut-targets-by-id/1kW1zea...,2c808cf51aedc2ba2be56afcefddbaac504706f43563ba...,282,365,166,116,0,True,False
3,source_based,train,/content/drive/.shortcut-targets-by-id/1kW1zea...,02ea0cc3c72010b44f38ac6a09d8121e079e4d97308e8b...,/content/drive/.shortcut-targets-by-id/1kW1zea...,58fbce591c39bb1290f8a0c62303322e55433ea2ad8b47...,1083,1708,684,399,0,True,False
4,source_based,val,/content/drive/.shortcut-targets-by-id/1kW1zea...,99d73ef8bba42c62819c9e21f23c17c787d01bb354e151...,/content/drive/.shortcut-targets-by-id/1kW1zea...,22dc1639165a134c31ccc4db70af2d15a67093c66a0fe2...,232,288,162,70,0,True,False
5,source_based,test,/content/drive/.shortcut-targets-by-id/1kW1zea...,d292fa8bc17ca12637b21059a38c195992765f7ce277ef...,/content/drive/.shortcut-targets-by-id/1kW1zea...,d8f1265e1158bd23f14f058fd0959b86f0f4ae83a7f9d6...,231,437,132,99,0,True,False


In [ ]:
#@title 5. Audit coral-only canonical data

rows = []

for protocol in PROTOCOLS:
    for split in SPLITS:
        path = CANONICAL_ROOT / protocol / split / "annotations.json"
        data = load_json(path)

        categories = {int(c["id"]): str(c["name"]) for c in data.get("categories", [])}
        image_ids = {int(i["id"]) for i in data.get("images", [])}
        annotation_ids = [int(a["id"]) for a in data.get("annotations", [])]
        category_ids = {int(a["category_id"]) for a in data.get("annotations", [])}

        unknown_image_refs = sum(
            int(a["image_id"]) not in image_ids
            for a in data.get("annotations", [])
        )

        invalid_boxes = 0
        for ann in data.get("annotations", []):
            x, y, w, h = [float(v) for v in ann["bbox"]]
            finite = bool(np.isfinite(np.asarray([x, y, w, h], dtype=float)).all())
            if (not finite) or w <= 0 or h <= 0:
                invalid_boxes += 1

        issues = []
        if categories != {1: "coral"}:
            issues.append("category mapping mismatch")
        if not category_ids.issubset({1}):
            issues.append("non-coral annotation remains")
        if len(annotation_ids) != len(set(annotation_ids)):
            issues.append("duplicate annotation IDs")
        if unknown_image_refs:
            issues.append("unknown image references")
        if invalid_boxes:
            issues.append("invalid bounding boxes")

        rows.append({
            "protocol": protocol,
            "split": split,
            "status": "PASS" if not issues else "FAIL",
            "issues": " | ".join(issues),
            "images": len(data.get("images", [])),
            "annotations": len(data.get("annotations", [])),
            "categories": json.dumps(categories, ensure_ascii=False),
            "unknown_image_refs": unknown_image_refs,
            "invalid_boxes": invalid_boxes,
            "sha256": sha256_file(path),
        })

audit_df = pd.DataFrame(rows)
save_csv(audit_df, MANIFESTS_ROOT / "coral_only_canonical_audit.csv")
display(audit_df)

if not (audit_df["status"] == "PASS").all():
    raise RuntimeError("Coral-only canonical audit gagal.")

,protocol,split,status,issues,images,annotations,categories,unknown_image_refs,invalid_boxes,sha256
0,time_based,train,PASS,,1061,1788,"{""1"": ""coral""}",0,0,9a69c8bef4df3eda69e73a174da95ee61df2a995f74e83...
1,time_based,val,PASS,,203,280,"{""1"": ""coral""}",0,0,c8a49530caf42326a71325eb133dea5eb5f2e2614ab8b2...
2,time_based,test,PASS,,282,365,"{""1"": ""coral""}",0,0,2c808cf51aedc2ba2be56afcefddbaac504706f43563ba...
3,source_based,train,PASS,,1083,1708,"{""1"": ""coral""}",0,0,58fbce591c39bb1290f8a0c62303322e55433ea2ad8b47...
4,source_based,val,PASS,,232,288,"{""1"": ""coral""}",0,0,22dc1639165a134c31ccc4db70af2d15a67093c66a0fe2...
5,source_based,test,PASS,,231,437,"{""1"": ""coral""}",0,0,d8f1265e1158bd23f14f058fd0959b86f0f4ae83a7f9d6...


In [ ]:
#@title 6. One-class COCOeval

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def evaluate_predictions(gt_path, predictions, temporary_name):
    temp = Path("/content") / temporary_name
    atomic_json(predictions, temp)

    coco_gt = COCO(str(gt_path))
    if sorted(coco_gt.getCatIds()) != [1]:
        raise RuntimeError("GT coral-only harus hanya memiliki category ID 1.")

    if not predictions:
        temp.unlink(missing_ok=True)
        return {
            "map50_95": 0.0,
            "map50": 0.0,
            "map75": 0.0,
            "coral_ap": 0.0,
            "detections": 0,
        }

    coco_dt = coco_gt.loadRes(str(temp))
    evaluator = COCOeval(coco_gt, coco_dt, "bbox")
    evaluator.params.imgIds = sorted(coco_gt.getImgIds())
    evaluator.params.catIds = [1]
    evaluator.params.maxDets = [1, 10, 100]

    buffer = StringIO()
    with redirect_stdout(buffer):
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    precision = evaluator.eval["precision"][:, :, 0, 0, -1]
    valid = precision[precision > -1]
    coral_ap = float(valid.mean()) if valid.size else float("nan")
    temp.unlink(missing_ok=True)

    return {
        "map50_95": float(evaluator.stats[0]),
        "map50": float(evaluator.stats[1]),
        "map75": float(evaluator.stats[2]),
        "coral_ap": coral_ap,
        "detections": len(predictions),
    }

In [ ]:
#@title 7. Shared image cache and YOLO labels

from google.colab import drive

def remount_drive():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    os.system("fusermount -u /content/drive 2>/dev/null")
    os.makedirs("/content/drive", exist_ok=True)
    drive.mount("/content/drive", force_remount=True)

def rsync_directory(source, destination, retries=3):
    source = Path(source)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    command = ["rsync", "-a", "--partial", "--inplace", f"{source}/", f"{destination}/"]

    error = None
    for attempt in range(1, retries + 1):
        try:
            subprocess.run(command, check=True)
            return
        except Exception as exc:
            error = exc
            print(f"rsync attempt {attempt} gagal:", repr(exc))
            if attempt < retries:
                remount_drive()
                time.sleep(3)
    raise RuntimeError(f"rsync gagal untuk {source}: {error}")

def relative_name(file_name):
    parts = list(Path(str(file_name).replace("\\", "/")).parts)
    while parts and parts[0].lower() in {"train", "val", "test", "images"}:
        parts = parts[1:]
    if not parts:
        raise ValueError(file_name)
    return Path(*parts)

def resolve_image(image_root, file_name):
    image_root = Path(image_root)
    candidates = [
        image_root / relative_name(file_name),
        image_root / Path(file_name).name,
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()

    matches = list(image_root.rglob(Path(file_name).name))
    if len(matches) == 1:
        return matches[0].resolve()
    if len(matches) > 1:
        raise RuntimeError(f"Ambiguous basename: {file_name}")
    raise FileNotFoundError(file_name)

def prepare_local_dataset(protocol):
    local_root = LOCAL_DATA / protocol

    for split in SPLITS:
        source_images = DATASET_ROOT / protocol / split / "images"
        common_images = LOCAL_IMAGES / protocol / split / "images"
        rsync_directory(source_images, common_images)

        scenario_images = local_root / split / "images"
        scenario_images.parent.mkdir(parents=True, exist_ok=True)

        if scenario_images.is_symlink():
            if scenario_images.resolve() != common_images.resolve():
                scenario_images.unlink()
        elif scenario_images.exists():
            shutil.rmtree(scenario_images)

        if not scenario_images.exists():
            scenario_images.symlink_to(common_images, target_is_directory=True)

        source_gt = CANONICAL_ROOT / protocol / split / "annotations.json"
        local_gt = local_root / split / "annotations.json"
        local_gt.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_gt, local_gt)

        data = load_json(local_gt)
        images = {int(i["id"]): i for i in data["images"]}
        anns_by_image = defaultdict(list)

        for ann in data["annotations"]:
            if int(ann["category_id"]) != 1:
                raise RuntimeError("Non-coral annotation found.")
            anns_by_image[int(ann["image_id"])].append(ann)

        label_root = LOCAL_IMAGES / protocol / split / "labels"
        shutil.rmtree(label_root, ignore_errors=True)
        label_root.mkdir(parents=True, exist_ok=True)
        (local_root / split / "labels.cache").unlink(missing_ok=True)

        for image_id, image in images.items():
            image_path = resolve_image(common_images, image["file_name"])
            relative = image_path.relative_to(common_images)
            label_path = (label_root / relative).with_suffix(".txt")
            label_path.parent.mkdir(parents=True, exist_ok=True)

            width = float(image["width"])
            height = float(image["height"])
            lines = []

            for ann in anns_by_image.get(image_id, []):
                x, y, w, h = [float(v) for v in ann["bbox"]]
                xc = (x + w / 2.0) / width
                yc = (y + h / 2.0) / height
                lines.append(f"0 {xc:.10f} {yc:.10f} {w/width:.10f} {h/height:.10f}")

            label_path.write_text("\n".join(lines), encoding="utf-8")

    yaml_path = local_root / f"coral_only_{protocol}.yaml"
    with yaml_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(
            {
                "path": str(local_root),
                "train": "train/images",
                "val": "val/images",
                "test": "test/images",
                "names": {0: "coral"},
            },
            f,
            sort_keys=False,
            allow_unicode=True,
        )

    return yaml_path

In [ ]:
#@title 8. Training helpers

import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def run_complete(run_dir, expected_epochs):
    run_dir = Path(run_dir)
    best = run_dir / "weights" / "best.pt"
    last = run_dir / "weights" / "last.pt"
    results = run_dir / "results.csv"

    if not all([best.exists(), last.exists(), results.exists()]):
        return False

    try:
        return len(pd.read_csv(results)) >= expected_epochs
    except Exception:
        return False

def selected_epoch(run_dir):
    results_path = Path(run_dir) / "results.csv"
    if not results_path.exists():
        return None

    results = pd.read_csv(results_path)
    if results.empty:
        return None

    results.columns = [str(c).strip() for c in results.columns]

    def norm(value):
        return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")

    candidates = []
    for column in results.columns:
        name = norm(column)
        if "map50_95" in name:
            priority = -2 if "metrics" in name else 0
            if name.endswith("_b") or "box" in name:
                priority -= 1
            candidates.append((priority, column))

    if not candidates:
        candidates = [
            (10, column)
            for column in results.columns
            if "map50" in norm(column) and "95" in norm(column)
        ]

    if not candidates:
        return None

    candidates.sort(key=lambda x: (x[0], x[1]))
    metric = pd.to_numeric(results[candidates[0][1]], errors="coerce").dropna()
    if metric.empty:
        return None

    best_index = metric.idxmax()
    epoch_column = next((c for c in results.columns if norm(c) == "epoch"), None)

    if epoch_column is not None:
        value = pd.to_numeric(
            pd.Series([results.loc[best_index, epoch_column]]),
            errors="coerce",
        ).iloc[0]
        if pd.notna(value):
            return int(value) + 1

    return int(results.index.get_loc(best_index)) + 1

environment = {
    "extension_tag": EXTENSION_TAG,
    "task": "coral_only",
    "model": MODEL_NAME,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "ultralytics": importlib_metadata.version("ultralytics"),
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "training_config": TRAIN_CONFIG,
    "inference_config": INFER_CONFIG,
}
atomic_json(environment, MANIFESTS_ROOT / "environment.json")
print(json.dumps(environment, indent=2))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{
  "extension_tag": "multiseed_wbf_results_ready_v3_extension_yolov10n_coral_only_v1",
  "task": "coral_only",
  "model": "yolov10n",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "ultralytics": "8.4.102",
  "cuda_available": true,
  "cuda_version": "12.8",
  "gpu": "NVIDIA L4",
  "training_config": {
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "workers": 0,
    "cache": false,
    "amp": true,
    "pretrained": true
  },
  "inference_config": {
    "conf": 0.001,
    "iou": 0.7,
    "max_det": 300,
    "imgsz": 640,
    "batch": 8,
    "augment": false
  }
}


In [ ]:
#@title 9. Train six coral-only runs

training_rows = []

if RUN_TRAINING:
    for protocol in PROTOCOLS:
        data_yaml = prepare_local_dataset(protocol)

        for seed in SEEDS:
            seed_all(seed)

            local_parent = LOCAL_RUNS / protocol / MODEL_NAME / f"seed_{seed}"
            local_train = local_parent / "train"
            drive_train = RUNS_ROOT / protocol / MODEL_NAME / f"seed_{seed}" / "train"

            if run_complete(drive_train, TRAIN_CONFIG["epochs"]):
                best = drive_train / "weights" / "best.pt"
                training_rows.append({
                    "protocol": protocol,
                    "seed": seed,
                    "status": "SKIP_COMPLETED",
                    "best_checkpoint": str(best),
                    "best_checkpoint_sha256": sha256_file(best),
                    "selected_epoch": selected_epoch(drive_train),
                })
                print("SKIP:", protocol, seed)
                continue

            if drive_train.exists() and not (local_train / "weights" / "last.pt").exists():
                rsync_directory(drive_train, local_train)

            last = local_train / "weights" / "last.pt"

            def sync_callback(trainer, destination=drive_train):
                try:
                    save_dir = Path(trainer.save_dir)
                    destination.mkdir(parents=True, exist_ok=True)
                    for relative in [
                        Path("weights/last.pt"),
                        Path("weights/best.pt"),
                        Path("results.csv"),
                        Path("args.yaml"),
                    ]:
                        source = save_dir / relative
                        target = destination / relative
                        if source.exists():
                            target.parent.mkdir(parents=True, exist_ok=True)
                            shutil.copy2(source, target)
                except Exception as exc:
                    print("Sync warning:", repr(exc))

            if last.exists():
                print("RESUME:", protocol, seed)
                model = YOLO(str(last))
                model.add_callback("on_model_save", sync_callback)
                model.train(resume=True)
            else:
                print("TRAIN:", protocol, seed)
                model = YOLO(PRETRAINED_WEIGHTS)
                model.add_callback("on_model_save", sync_callback)
                model.train(
                    data=str(data_yaml),
                    epochs=TRAIN_CONFIG["epochs"],
                    imgsz=TRAIN_CONFIG["imgsz"],
                    batch=TRAIN_CONFIG["batch"],
                    seed=seed,
                    deterministic=True,
                    device=DEVICE,
                    workers=TRAIN_CONFIG["workers"],
                    cache=TRAIN_CONFIG["cache"],
                    amp=TRAIN_CONFIG["amp"],
                    pretrained=TRAIN_CONFIG["pretrained"],
                    project=str(local_parent),
                    name="train",
                    exist_ok=True,
                    save=True,
                    plots=True,
                    verbose=True,
                )

            rsync_directory(local_train, drive_train)

            if not run_complete(drive_train, TRAIN_CONFIG["epochs"]):
                raise RuntimeError(f"Training belum lengkap: {protocol}, seed {seed}")

            best = drive_train / "weights" / "best.pt"
            training_rows.append({
                "protocol": protocol,
                "seed": seed,
                "status": "TRAINED",
                "best_checkpoint": str(best),
                "best_checkpoint_sha256": sha256_file(best),
                "selected_epoch": selected_epoch(drive_train),
                "gpu": environment["gpu"],
            })

            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

training_df = pd.DataFrame(training_rows)
if not training_df.empty:
    save_csv(training_df, REPORTS_ROOT / "training_summary.csv")
    display(training_df)

TRAIN: time_based 42
New https://pypi.org/project/ultralytics/8.4.113 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/multiseed_wbf_results_ready_v3_extension_yolov10n_coral_only_v1_data/time_based/coral_only_time_based.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, l

,protocol,seed,status,best_checkpoint,best_checkpoint_sha256,selected_epoch,gpu
0,time_based,42,TRAINED,/content/drive/.shortcut-targets-by-id/1kW1zea...,0ce465154c0ab934e9d46ccff62fc8cc7887bd636a95a3...,86,NVIDIA L4
1,time_based,123,TRAINED,/content/drive/.shortcut-targets-by-id/1kW1zea...,86503cfabca7e76b4cab7216fe058069f1aaef80b2fd9e...,84,NVIDIA L4
2,time_based,2026,TRAINED,/content/drive/.shortcut-targets-by-id/1kW1zea...,dec01c12bed400597854fb6a886ca36d7e3c7b99557989...,68,NVIDIA L4
3,source_based,42,TRAINED,/content/drive/.shortcut-targets-by-id/1kW1zea...,3bc2524d900b0a964dde64cf7c491f7618cffdccd72ed4...,2,NVIDIA L4
4,source_based,123,TRAINED,/content/drive/.shortcut-targets-by-id/1kW1zea...,080951bcafe631f2575d6847146ca62eaf89e35b513241...,2,NVIDIA L4
5,source_based,2026,TRAINED,/content/drive/.shortcut-targets-by-id/1kW1zea...,497c78dd1feaf44b0600099b6e6258530034adb8dff0be...,2,NVIDIA L4


In [ ]:
#@title 10. Inference and evaluation

def image_records(local_root, split):
    data = load_json(Path(local_root) / split / "annotations.json")
    return sorted(data["images"], key=lambda x: int(x["id"]))

def image_paths(local_root, split):
    local_root = Path(local_root)
    records = image_records(local_root, split)
    root = local_root / split / "images"
    paths = [resolve_image(root, record["file_name"]) for record in records]
    return records, paths

def predict_coral(model, local_root, split):
    records, paths = image_paths(local_root, split)
    results = model.predict(
        source=[str(path) for path in paths],
        stream=True,
        batch=INFER_CONFIG["batch"],
        imgsz=INFER_CONFIG["imgsz"],
        conf=INFER_CONFIG["conf"],
        iou=INFER_CONFIG["iou"],
        max_det=INFER_CONFIG["max_det"],
        augment=INFER_CONFIG["augment"],
        verbose=False,
        device=DEVICE,
    )

    predictions = []
    count = 0

    for count, result in enumerate(results, start=1):
        image_id = int(records[count - 1]["id"])
        if result.boxes is None:
            continue

        boxes = result.boxes.xyxy.detach().cpu().numpy()
        scores = result.boxes.conf.detach().cpu().numpy()
        classes = result.boxes.cls.detach().cpu().numpy().astype(int)

        for box, score, class_index in zip(boxes, scores, classes):
            class_index = int(class_index)
            class_name = str(model.names[class_index]).strip().lower()
            if class_index != 0 or class_name != "coral":
                raise RuntimeError(f"Unexpected model class: {class_index}, {class_name}")

            x1, y1, x2, y2 = [float(v) for v in box]
            predictions.append({
                "image_id": image_id,
                "category_id": 1,
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "score": float(score),
            })

    if count != len(records):
        raise RuntimeError(f"Prediction count mismatch: {count} vs {len(records)}")

    return predictions

result_rows = []
checkpoint_rows = []

if RUN_EVALUATION:
    for protocol in PROTOCOLS:
        data_yaml = prepare_local_dataset(protocol)
        local_root = data_yaml.parent

        for seed in SEEDS:
            run_dir = RUNS_ROOT / protocol / MODEL_NAME / f"seed_{seed}" / "train"
            checkpoint = run_dir / "weights" / "best.pt"
            if not checkpoint.exists():
                raise FileNotFoundError(checkpoint)

            checkpoint_hash = sha256_file(checkpoint)
            epoch = selected_epoch(run_dir)

            checkpoint_rows.append({
                "protocol": protocol,
                "model": MODEL_NAME,
                "seed": seed,
                "checkpoint_path": str(checkpoint),
                "checkpoint_sha256": checkpoint_hash,
                "selected_epoch": epoch,
            })

            model = YOLO(str(checkpoint))

            for split in ["val", "test"]:
                output_dir = PRED_ROOT / protocol / MODEL_NAME / f"seed_{seed}"
                output_dir.mkdir(parents=True, exist_ok=True)

                prediction_path = output_dir / f"{split}.json"
                manifest_path = output_dir / f"{split}_manifest.json"
                gt_path = CANONICAL_ROOT / protocol / split / "annotations.json"

                predictions = predict_coral(model, local_root, split)
                atomic_json(predictions, prediction_path)

                manifest = {
                    "extension_tag": EXTENSION_TAG,
                    "task": "coral_only",
                    "protocol": protocol,
                    "model": MODEL_NAME,
                    "seed": seed,
                    "split": split,
                    "checkpoint_path": str(checkpoint),
                    "checkpoint_sha256": checkpoint_hash,
                    "gt_path": str(gt_path),
                    "gt_sha256": sha256_file(gt_path),
                    "inference_config": INFER_CONFIG,
                    "prediction_path": str(prediction_path),
                    "prediction_sha256": sha256_file(prediction_path),
                }
                atomic_json(manifest, manifest_path)

                metrics = evaluate_predictions(
                    gt_path,
                    predictions,
                    f"coral_only_{protocol}_{seed}_{split}.json",
                )

                result_rows.append({
                    "extension_tag": EXTENSION_TAG,
                    "task": "coral_only",
                    "protocol": protocol,
                    "model": MODEL_NAME,
                    "seed": seed,
                    "split": split,
                    "selected_epoch": epoch,
                    "checkpoint_path": str(checkpoint),
                    "checkpoint_sha256": checkpoint_hash,
                    "prediction_path": str(prediction_path),
                    "prediction_sha256": sha256_file(prediction_path),
                    "gt_path": str(gt_path),
                    "gt_sha256": sha256_file(gt_path),
                    **metrics,
                })

            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

checkpoint_df = pd.DataFrame(checkpoint_rows)
results_df = pd.DataFrame(result_rows)
save_csv(checkpoint_df, MANIFESTS_ROOT / "checkpoint_manifest.csv")
save_csv(results_df, REPORTS_ROOT / "results_by_seed.csv")
display(checkpoint_df)
display(results_df)

In [ ]:
#@title 11. Summary and completion gate

checkpoint_df = pd.read_csv(MANIFESTS_ROOT / "checkpoint_manifest.csv")
results_df = pd.read_csv(REPORTS_ROOT / "results_by_seed.csv")

summary_rows = []

for (protocol, model_name, split), group in results_df.groupby(
    ["protocol", "model", "split"]
):
    row = {
        "protocol": protocol,
        "model": model_name,
        "split": split,
        "n_seeds": group["seed"].nunique(),
    }

    for metric in ["map50_95", "map50", "map75", "coral_ap", "detections"]:
        values = pd.to_numeric(group[metric], errors="coerce").dropna().to_numpy(float)
        row[f"{metric}_mean"] = float(np.mean(values))
        row[f"{metric}_sd"] = sample_sd(values)
        row[f"{metric}_min"] = float(np.min(values))
        row[f"{metric}_max"] = float(np.max(values))

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
save_csv(summary_df, REPORTS_ROOT / "multiseed_summary.csv")

val_df = results_df[results_df["split"] == "val"]
test_df = results_df[results_df["split"] == "test"]
expected = len(PROTOCOLS) * len(SEEDS)

completion = {
    "extension_tag": EXTENSION_TAG,
    "task": "coral_only",
    "expected_checkpoints": expected,
    "actual_checkpoints": len(checkpoint_df),
    "expected_validation_results": expected,
    "actual_validation_results": len(val_df),
    "expected_test_results": expected,
    "actual_test_results": len(test_df),
    "selected_epoch_complete": bool(
        pd.to_numeric(checkpoint_df["selected_epoch"], errors="coerce").notna().all()
    ),
    "checkpoint_files_present": bool(
        checkpoint_df["checkpoint_path"].map(lambda value: Path(value).exists()).all()
    ),
    "prediction_files_present": bool(
        results_df["prediction_path"].map(lambda value: Path(value).exists()).all()
    ),
    "exact_protocols": set(checkpoint_df["protocol"].astype(str)) == set(PROTOCOLS),
    "exact_seeds": set(checkpoint_df["seed"].astype(int)) == set(SEEDS),
    "writes_to_parent_v3": False,
    "images_duplicated_in_drive": False,
}

completion["ready_for_reporting"] = all([
    completion["actual_checkpoints"] == expected,
    completion["actual_validation_results"] == expected,
    completion["actual_test_results"] == expected,
    completion["selected_epoch_complete"],
    completion["checkpoint_files_present"],
    completion["prediction_files_present"],
    completion["exact_protocols"],
    completion["exact_seeds"],
])

atomic_json(completion, MANIFESTS_ROOT / "extension_completion.json")
display(summary_df)
print(json.dumps(completion, indent=2))

if not completion["ready_for_reporting"]:
    raise RuntimeError("Coral-only extension belum lengkap.")

,protocol,model,split,n_seeds,map50_95_mean,map50_95_sd,map50_95_min,map50_95_max,map50_mean,map50_sd,...,map75_min,map75_max,coral_ap_mean,coral_ap_sd,coral_ap_min,coral_ap_max,detections_mean,detections_sd,detections_min,detections_max
0,source_based,yolov10n,test,3,0.615806,0.006076,0.610368,0.622363,0.823509,0.009332,...,0.707522,0.723522,0.615806,0.006076,0.610368,0.622363,4362.666667,740.730270,3550.0,5000.0
1,source_based,yolov10n,val,3,0.748704,0.019318,0.736592,0.770983,0.912399,0.010113,...,0.821676,0.847354,0.748704,0.019318,0.736592,0.770983,2390.666667,674.715743,1860.0,3150.0
2,time_based,yolov10n,test,3,0.489625,0.007353,0.481470,0.495750,0.671497,0.011216,...,0.517067,0.553002,0.489625,0.007353,0.481470,0.495750,1374.000000,305.235974,1147.0,1721.0
3,time_based,yolov10n,val,3,0.581938,0.005135,0.576272,0.586286,0.784371,0.006508,...,0.612576,0.628636,0.581938,0.005135,0.576272,0.586286,989.000000,260.781518,831.0,1290.0


{
  "extension_tag": "multiseed_wbf_results_ready_v3_extension_yolov10n_coral_only_v1",
  "task": "coral_only",
  "expected_checkpoints": 6,
  "actual_checkpoints": 6,
  "expected_validation_results": 6,
  "actual_validation_results": 6,
  "expected_test_results": 6,
  "actual_test_results": 6,
  "selected_epoch_complete": true,
  "checkpoint_files_present": true,
  "prediction_files_present": true,
  "exact_protocols": true,
  "exact_seeds": true,
  "writes_to_parent_v3": false,
  "images_duplicated_in_drive": false,
  "ready_for_reporting": true
}


In [ ]:
#@title 12. Optional two-class coral-AP comparison

if RUN_OPTIONAL_TWO_CLASS_COMPARISON:
    two_class_path = TWO_CLASS_ROOT / "reports" / "yolov10n_results_by_seed.csv"

    if not two_class_path.exists():
        print("Two-class extension result belum tersedia. Comparison skipped.")
    else:
        two_class = pd.read_csv(two_class_path)
        two_class = two_class[two_class["split"] == "test"][
            ["protocol", "seed", "coral_ap"]
        ].rename(columns={"coral_ap": "two_class_coral_ap"})

        coral_only = results_df[results_df["split"] == "test"][
            ["protocol", "seed", "coral_ap"]
        ].rename(columns={"coral_ap": "coral_only_coral_ap"})

        comparison = two_class.merge(
            coral_only,
            on=["protocol", "seed"],
            how="inner",
            validate="one_to_one",
        )

        comparison["coral_ap_delta_coral_only_minus_two_class"] = (
            comparison["coral_only_coral_ap"]
            - comparison["two_class_coral_ap"]
        )

        summary_rows = []
        for protocol, group in comparison.groupby("protocol"):
            delta = group["coral_ap_delta_coral_only_minus_two_class"].to_numpy(float)
            summary_rows.append({
                "protocol": protocol,
                "n_seeds": len(group),
                "two_class_coral_ap_mean": float(group["two_class_coral_ap"].mean()),
                "coral_only_coral_ap_mean": float(group["coral_only_coral_ap"].mean()),
                "coral_ap_delta_mean": float(np.mean(delta)),
                "coral_ap_delta_sd": sample_sd(delta),
            })

        comparison_summary = pd.DataFrame(summary_rows)
        save_csv(comparison, REPORTS_ROOT / "two_class_vs_coral_only_by_seed.csv")
        save_csv(
            comparison_summary,
            REPORTS_ROOT / "two_class_vs_coral_only_summary.csv",
        )
        display(comparison)
        display(comparison_summary)

In [18]:
#@title 13. Optional FPS

from PIL import Image

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def spaced_indices(total, desired):
    if total <= desired:
        return list(range(total))
    return sorted(set(np.linspace(0, total - 1, num=desired, dtype=int).tolist()))

fps_rows = []

if RUN_OPTIONAL_FPS:
    for protocol in PROTOCOLS:
        data_yaml = prepare_local_dataset(protocol)
        local_root = data_yaml.parent
        _, paths = image_paths(local_root, "test")
        indices = spaced_indices(len(paths), FPS_IMAGES)

        arrays = []
        for index in indices:
            with Image.open(paths[index]) as image:
                arrays.append(np.asarray(image.convert("RGB")).copy())

        checkpoint = (
            RUNS_ROOT
            / protocol
            / MODEL_NAME
            / f"seed_{FPS_SEED}"
            / "train"
            / "weights"
            / "best.pt"
        )
        model = YOLO(str(checkpoint))

        for array in arrays[: min(FPS_WARMUP, len(arrays))]:
            model.predict(
                source=array,
                imgsz=TRAIN_CONFIG["imgsz"],
                conf=INFER_CONFIG["conf"],
                iou=INFER_CONFIG["iou"],
                max_det=INFER_CONFIG["max_det"],
                augment=False,
                verbose=False,
                device=DEVICE,
            )

        sync_cuda()

        for repeat in range(1, FPS_REPEATS + 1):
            latencies = []

            for array in arrays:
                sync_cuda()
                start = time.perf_counter()
                model.predict(
                    source=array,
                    imgsz=TRAIN_CONFIG["imgsz"],
                    conf=INFER_CONFIG["conf"],
                    iou=INFER_CONFIG["iou"],
                    max_det=INFER_CONFIG["max_det"],
                    augment=False,
                    verbose=False,
                    device=DEVICE,
                )
                sync_cuda()
                latencies.append(time.perf_counter() - start)

            latency = np.asarray(latencies, dtype=float)
            fps_rows.append({
                "protocol": protocol,
                "model": MODEL_NAME,
                "task": "coral_only",
                "seed": FPS_SEED,
                "repeat": repeat,
                "images": len(arrays),
                "fps": float(1.0 / latency.mean()),
                "mean_latency_ms": float(latency.mean() * 1000.0),
                "p95_latency_ms": float(np.percentile(latency * 1000.0, 95)),
                "parameter_count": int(
                    sum(parameter.numel() for parameter in model.model.parameters())
                ),
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            })

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

fps_df = pd.DataFrame(fps_rows)

if not fps_df.empty:
    save_csv(fps_df, REPORTS_ROOT / "fps_repeated_results.csv")
    display(fps_df)
else:
    print("FPS skipped. Jalankan nanti pada GPU benchmark yang sama dengan V3.")

,protocol,model,task,seed,repeat,images,fps,mean_latency_ms,p95_latency_ms,parameter_count,gpu
0,time_based,yolov10n,coral_only,42,1,100,81.062254,12.336198,14.354950,2265363,NVIDIA L4
1,time_based,yolov10n,coral_only,42,2,100,81.619534,12.251969,12.933872,2265363,NVIDIA L4
2,time_based,yolov10n,coral_only,42,3,100,81.372633,12.289144,12.861593,2265363,NVIDIA L4
3,source_based,yolov10n,coral_only,42,1,100,83.267222,12.009528,12.587982,2265363,NVIDIA L4
4,source_based,yolov10n,coral_only,42,2,100,82.846760,12.070478,12.584851,2265363,NVIDIA L4
5,source_based,yolov10n,coral_only,42,3,100,82.871203,12.066918,12.654892,2265363,NVIDIA L4


In [17]:
#@title 14. Final report and evidence ZIP

completion = load_json(MANIFESTS_ROOT / "extension_completion.json")
summary_df = pd.read_csv(REPORTS_ROOT / "multiseed_summary.csv")
test_summary = summary_df[summary_df["split"] == "test"]

lines = [
    "# AU-FISH YOLOv10n Coral-Only Ablation",
    "",
    f"Extension tag: `{EXTENSION_TAG}`",
    "",
    f"Ready for reporting: **{completion['ready_for_reporting']}**",
    "",
    "## Storage",
    "",
    "- V3 is read-only.",
    "- No image folder is duplicated in Google Drive.",
    "- All original images are retained.",
    "- Fish annotations are removed.",
    "- Images without coral are negative images.",
    "",
    "## Results",
    "",
]

for row in test_summary.itertuples():
    lines.append(
        f"- {row.protocol}: one-class mAP50-95 {row.map50_95_mean:.6f} "
        f"± {row.map50_95_sd:.6f}; coral AP {row.coral_ap_mean:.6f} "
        f"± {row.coral_ap_sd:.6f}."
    )

lines.extend([
    "",
    "## Reporting rule",
    "",
    "- Compare coral AP between the two-class and coral-only models.",
    "- Do not directly equate one-class mAP50-95 with two-class mAP50-95.",
    "- Measure FPS on the same device and timing boundary used for V3.",
])

report_text = "\n".join(lines)
(REPORTS_ROOT / "CORAL_ONLY_ABLATION_REPORT.md").write_text(
    report_text,
    encoding="utf-8",
)

evidence_zip = EXT_ROOT / "CORAL_ONLY_ABLATION_EVIDENCE.zip"
evidence_zip.unlink(missing_ok=True)

with zipfile.ZipFile(evidence_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for root in [CONFIG_ROOT, CANONICAL_ROOT, MANIFESTS_ROOT, PRED_ROOT, REPORTS_ROOT]:
        for path in root.rglob("*"):
            if path.is_file():
                archive.write(path, arcname=str(path.relative_to(EXT_ROOT)))

print(report_text)
print("\nEvidence ZIP:", evidence_zip)

# AU-FISH YOLOv10n Coral-Only Ablation

Extension tag: `multiseed_wbf_results_ready_v3_extension_yolov10n_coral_only_v1`

Ready for reporting: **True**

## Storage

- V3 is read-only.
- No image folder is duplicated in Google Drive.
- All original images are retained.
- Fish annotations are removed.
- Images without coral are negative images.

## Results

- source_based: one-class mAP50-95 0.615806 ± 0.006076; coral AP 0.615806 ± 0.006076.
- time_based: one-class mAP50-95 0.489625 ± 0.007353; coral AP 0.489625 ± 0.007353.

## Reporting rule

- Compare coral AP between the two-class and coral-only models.
- Do not directly equate one-class mAP50-95 with two-class mAP50-95.
- Measure FPS on the same device and timing boundary used for V3.

Evidence ZIP: /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_yolov10n_coral_only_v1/CORAL_ONLY_ABLATION_EVIDENCE.zip
